# Real Madrid AI Transfer Intelligence
## Notebook 01 — Data Collection

**Goal:** collect player-season statistics for Real Madrid in La Liga for 2023/24, 2024/25 and 2025/26, preserve source provenance, and export a clean first data layer.

**Source:** FBref via the documented `soccerdata.FBref` interface. The notebook retrieves multiple statistical tables and merges them by player, squad, competition and season where possible.

**Important scope rules**
- This notebook collects domestic league data only.
- Each season remains a separate observation.
- Yan Diomande's career history is not inserted into the historical Real Madrid squad. He only appears in Real Madrid data if he appears in the 2026/27 scope, which is outside this three-season audit.
- Missing values remain missing; they are not converted to zero.
- Scraping may be rate-limited or blocked by FBref. If that happens, use the manual CSV fallback described at the end and record the actual source file.


## 1. Setup

Run the notebook from the project root. Install dependencies once using `requirements.txt`. The scraper caches downloaded data locally to reduce repeated requests.

In [1]:
from pathlib import Path
import re
import json
import pandas as pd
import numpy as np
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    # Supports launching Jupyter from the notebooks/ folder.
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SEASONS = ["2023-2024", "2024-2025", "2025-2026"]
LEAGUE = "ESP-La Liga"
CLUB_PATTERN = r"real madrid"
SOURCE_NAME = "FBref via soccerdata"
RETRIEVED_AT_UTC = datetime.now(timezone.utc).isoformat()

print("Project root:", PROJECT_ROOT)
print("Seasons:", SEASONS)
print("League:", LEAGUE)


Project root: C:\Users\1alma\DataspellProjects\real-madrid-ai
Seasons: ['2023-2024', '2024-2025', '2025-2026']
League: ESP-La Liga


## 2. Install and initialize FBref reader

The API call below follows the current SoccerData FBref documentation. If the source changes its table structure, the notebook saves the raw table before transformation so the data can be inspected and mapped without silently dropping fields.

In [2]:
# If needed, run this once in a notebook cell:
# %pip install -r requirements.txt

import soccerdata as sd

fbref = sd.FBref(
    leagues=LEAGUE,
    seasons=SEASONS,
    data_dir=RAW_DIR / "fbref_cache",
)
print("SoccerData version:", getattr(sd, "__version__", "not exposed"))


## 3. Download player statistics

We request separate FBref table families. This gives us a modular dataset and makes it easier to document which metric came from which table. Some fields may be absent for some player positions; that is expected.

In [3]:
STAT_TYPES = ["standard", "shooting", "playing_time", "misc", "keeper"]
raw_tables = {}

for stat_type in STAT_TYPES:
    print(f"Downloading {stat_type} ...")
    table = fbref.read_player_season_stats(stat_type=stat_type)
    raw_tables[stat_type] = table.copy()
    output_path = RAW_DIR / f"fbref_player_{stat_type}.parquet"
    table.to_parquet(output_path)
    print(f"  saved {len(table):,} rows -> {output_path.name}")


## 4. Inspect and normalize table structure

SoccerData may return a MultiIndex or a regular index depending on source/version. We flatten the index and column labels, then identify the club column. The function intentionally raises an informative error if the team field cannot be identified rather than silently selecting the wrong rows.

In [4]:
def flatten_fbref_table(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if isinstance(out.columns, pd.MultiIndex):
        out.columns = [
            "_".join(str(part) for part in col if str(part) not in ("", "None"))
            for col in out.columns.to_list()
        ]
    else:
        out.columns = [str(c) for c in out.columns]

    if isinstance(out.index, pd.MultiIndex):
        out = out.reset_index()
    elif out.index.name is not None:
        out = out.reset_index()

    out.columns = [re.sub(r"\s+", "_", str(c).strip()).lower() for c in out.columns]
    return out

def find_team_column(df: pd.DataFrame) -> str:
    candidates = [c for c in df.columns if c in {"team", "squad", "team_name", "squad_name"}]
    if not candidates:
        candidates = [
            c for c in df.columns
            if df[c].astype(str).str.contains(CLUB_PATTERN, case=False, regex=True, na=False).any()
        ]
    if not candidates:
        raise ValueError(
            "Could not identify the Real Madrid team column. "
            f"Available columns: {df.columns.tolist()}"
        )
    return candidates[0]

standard_flat = flatten_fbref_table(raw_tables["standard"])
team_col = find_team_column(standard_flat)
print("Detected team column:", team_col)
display(standard_flat.head())


,league,season,team,player,nation,pos,age,born,playing_time_mp,playing_time_starts,...,performance_g-pk,performance_pk,performance_pkatt,performance_crdy,performance_crdr,per_90_minutes_gls,per_90_minutes_ast,per_90_minutes_g+a,per_90_minutes_g-pk,per_90_minutes_g+a-pk
0,ESP-La Liga,2324,Alavés,Abde Rebbach,ALG,MF,24,1998,23,6,...,1,0,0,1,0,0.12,0.0,0.12,0.12,0.12
1,ESP-La Liga,2324,Alavés,Abdel Abqar,MAR,DF,24,1999,27,27,...,0,0,0,10,0,0.0,0.0,0.0,0.0,0.0
2,ESP-La Liga,2324,Alavés,Aleksandar Sedlar,SRB,DF,31,1991,11,11,...,1,0,0,2,0,0.1,0.0,0.1,0.1,0.1
3,ESP-La Liga,2324,Alavés,Alex Sola,ESP,MF,24,1999,29,14,...,0,0,0,2,0,0.0,0.21,0.21,0.0,0.21
4,ESP-La Liga,2324,Alavés,Ander Guevara,ESP,MF,26,1997,37,33,...,1,0,0,7,0,0.03,0.0,0.03,0.03,0.03


## 5. Filter Real Madrid and join metric families

The join keys are inferred from common identifier fields. The notebook checks for duplicate join keys and reports them. We keep standard statistics as the base table; supplemental metrics are joined only when their keys are available.

In [5]:
def real_madrid_only(df: pd.DataFrame) -> pd.DataFrame:
    flat = flatten_fbref_table(df)
    team_field = find_team_column(flat)
    mask = flat[team_field].astype(str).str.contains(CLUB_PATTERN, case=False, regex=True, na=False)
    return flat.loc[mask].copy()

filtered = {name: real_madrid_only(table) for name, table in raw_tables.items()}

base = filtered["standard"].copy()
print("Standard table rows for Real Madrid:", len(base))
display(base.head(10))

def candidate_join_keys(left: pd.DataFrame, right: pd.DataFrame):
    preferred = ["player", "squad", "team", "comp", "season"]
    return [c for c in preferred if c in left.columns and c in right.columns]

merged = base
merge_log = []

for stat_type in ["shooting", "playing_time", "misc", "keeper"]:
    supplement = filtered[stat_type]
    keys = candidate_join_keys(merged, supplement)
    if "player" not in keys:
        merge_log.append({"table": stat_type, "status": "skipped", "reason": "player key missing", "keys": keys})
        continue

    # Avoid bringing in duplicate non-key columns from prior table families.
    duplicate_nonkeys = [c for c in supplement.columns if c in merged.columns and c not in keys]
    supplement = supplement.drop(columns=duplicate_nonkeys, errors="ignore")

    duplicate_keys = supplement.duplicated(keys).sum()
    if duplicate_keys:
        merge_log.append({
            "table": stat_type, "status": "skipped",
            "reason": f"{duplicate_keys} duplicate rows on keys {keys}", "keys": keys
        })
        continue

    merged = merged.merge(supplement, on=keys, how="left", validate="one_to_one")
    merge_log.append({"table": stat_type, "status": "merged", "reason": "", "keys": keys})

pd.DataFrame(merge_log)


,table,status,reason,keys
0,shooting,merged,,"[player, team, season]"
1,playing_time,merged,,"[player, team, season]"
2,misc,merged,,"[player, team, season]"
3,keeper,merged,,"[player, team, season]"


## 6. Data-quality checks

We validate the resulting table before exporting. The expected output is one row per player-season-competition for Real Madrid. If the source has duplicate rows or missing core identifiers, stop and inspect the raw extracts.

In [6]:
# Normalize key labels when present.
rename_map = {}
for col in merged.columns:
    if col in {"player", "squad", "comp", "season"}:
        rename_map[col] = {"player":"player_name", "squad":"team", "comp":"competition", "season":"season"}[col]
merged = merged.rename(columns=rename_map)

required = ["player_name", "season"]
missing_required = [c for c in required if c not in merged.columns]
if missing_required:
    raise ValueError(f"Missing required columns after normalization: {missing_required}")

# Preserve source values and add provenance.
merged["source"] = SOURCE_NAME
merged["retrieved_at_utc"] = RETRIEVED_AT_UTC
merged["scope"] = "La Liga"

# Keep only actual player rows; FBref tables can include aggregate rows.
merged = merged[
    merged["player_name"].notna()
    & ~merged["player_name"].astype(str).str.contains(r"^squad total$|^opponent total$", case=False, regex=True)
].copy()

duplicate_key_cols = [c for c in ["player_name", "season", "competition"] if c in merged.columns]
duplicate_count = merged.duplicated(duplicate_key_cols).sum()
missing_counts = merged.isna().sum().sort_values(ascending=False)

quality_summary = pd.DataFrame({
    "check": ["rows", "unique players", "duplicate player-season rows", "missing player names", "missing seasons"],
    "value": [
        len(merged),
        merged["player_name"].nunique(),
        int(duplicate_count),
        int(merged["player_name"].isna().sum()),
        int(merged["season"].isna().sum()),
    ]
})
display(quality_summary)
display(missing_counts.head(20).rename("missing_values").to_frame())

if duplicate_count:
    display(merged.loc[merged.duplicated(duplicate_key_cols, keep=False)].sort_values(duplicate_key_cols))


,missing_values
performance_pkcon,92
performance_pkwon,92
penalty_kicks_save%,89
penalty_kicks_pka,84
performance_l,84
performance_ga90,84
performance_saves,84
performance_save%,84
performance_sota,84
performance_d,84


## 7. Export processed data

We save the combined player-season file and a canonical player dimension. The player dimension is derived from the data; the fact table remains the source of season-specific performance.

In [9]:
player_season_path = PROCESSED_DIR / "real_madrid_player_season_laliga_2023_2026.csv"
players_path = PROCESSED_DIR / "real_madrid_players.csv"

merged.to_csv(player_season_path, index=False)

players = (
    merged[["player_name"]]
    .drop_duplicates()
    .sort_values("player_name")
    .reset_index(drop=True)
)
players["player_id"] = (
    players["player_name"].str.lower()
    .str.normalize("NFKD")
    .str.encode("ascii", errors="ignore")
    .str.decode("utf-8")
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)
players = players[["player_id", "player_name"]]
players.to_csv(players_path, index=False)

print("Saved:", player_season_path)
print("Saved:", players_path)
print("Player-seasons:", len(merged), "| unique players:", len(players))


## 8. Create a source and coverage manifest

This manifest records the source, retrieval timestamp, season coverage and output files. It is part of the project's audit trail.

In [10]:
manifest = {
    "source": SOURCE_NAME,
    "source_docs": "https://soccerdata.readthedocs.io/en/stable/reference/fbref.html",
    "fbref_league": LEAGUE,
    "seasons_requested": SEASONS,
    "scope": "Real Madrid first team, La Liga only",
    "retrieved_at_utc": RETRIEVED_AT_UTC,
    "rows_exported": int(len(merged)),
    "unique_players": int(merged["player_name"].nunique()),
    "duplicate_player_season_rows": int(duplicate_count),
    "files": [str(player_season_path), str(players_path)],
    "limitations": [
        "FBref may block or rate-limit automated requests.",
        "Coverage depends on the source's player-season tables.",
        "This extract is domestic league only, not all competitions.",
        "Missing metrics can be position-specific or unavailable from source."
    ]
}
manifest_path = PROCESSED_DIR / "collection_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(manifest_path)
